1. Installing libraries


In [ ]:
# Core ML libraries
!pip install -q torch==2.9.0
!pip install -q transformers==4.57.1
# Web search
!pip install -q ddgs==9.4.0
# LangChain stack
!pip install -q langchain==0.3.26
!pip install -q langchain-openai==0.3.27
!pip install -q langchain-community==0.3.27
# OpenAI client
!pip install -q openai==1.86.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 841.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.9.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages

1.2 install  ollama and deepseek-r1


In [ ]:
# Install Ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 67 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (1,739 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current 

In [ ]:
# Start Ollama server in background
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

In [ ]:
# Pull both models
!ollama pull llama3.2:3b
!ollama pull deepseek-r1:8b

In [ ]:
!ollama list

NAME              ID              SIZE      MODIFIED           
deepseek-r1:8b    6995872bfe4c    5.2 GB    5 seconds ago         
llama3.2:3b       a80c4f17acd5    2.0 GB    About a minute ago    


2. Inference time scaling

2.1 Few shot COT

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")

few_shot_example = """Q: if it is 3pm in londoan(UTC+0), what time is in NEW YORK (UTC-5)?
A: London is 5 hours ahead ,so we subtract 5 , the final answer in 10 Am.

Q: a tank holds 60 L of water. It leaks 3 l per hour and is filed at 5l per hour. how much water after 4 h?
A: Net fill = 5-3 = 2L/h. 2*4 = 8l. fial answer is 68 L.
"""

question = " a rectangle has perimeter 40cm width 5cm . what is length?"

prompt = few_shot_example + f"Q:{question} \n A:"
model = "llama3.2:3b"

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user","content":prompt}],
    temperature=0.3
)
print(response.choices[0].message.content)

I see you have three questions!

Let's solve them one by one:

1. London (UTC+0) to New York (UTC-5):

You're correct that London is 5 hours ahead of New York. To find the time in New York, you subtract 5 hours from 3 pm:

3 pm - 5 hours = 10 am

So, the answer is 10 am.

2. Tank problem:

You're correct that the net fill rate is 5 L/h - 3 L/h = 2 L/h. To find the total amount of water after 4 hours, you multiply the net fill rate by the number of hours:

2 L/h x 4 h = 8 L

However, you also need to consider the initial amount of water (60 L) and the amount leaked (3 L/h x 4 h = 12 L). The correct calculation is:

Initial water - leaked water + net fill = 60 L - 12 L + 8 L = 56 L

So, the final answer is 56 L, not 68 L.

3. Rectangle problem:

To find the length of the rectangle, you can use the formula for the perimeter:

Perimeter = 2(Length + Width)

You're given the perimeter (40 cm) and the width (5 cm). Let's plug in the values:

40 = 2(Length + 5)

Simplify the equation:

40 = 2

2.2 zero shot COT

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")

question = "why so we use neural networks to build LLMs?"

prompt = f"You are knowledgabke tutor. Answer the question. Question: {question} \n Lets's think step by step."

model = "llama3.2:3b"

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user","content":prompt}],
    temperature= 1
)
print(response.choices[0].message.content)


To understand why neural networks are used to build Large Language Models (LLMs), let's break it down step by step.

**What are Large Language Models (LLMs)?**

LLMs are a type of artificial intelligence (AI) designed to process and understand natural language data. They aim to generate text or translate languages by analyzing vast amounts of text data.

**How do LLMs work?**

To build an LLM, a complex computational architecture is required, which needs to:

1. Process and represent complex text patterns
2. Analyze vast amounts of text data to learn patterns and relationships
3. Generate text based on this learned knowledge

**What are the challenges in building LLMs?**

The main challenges in building LLMs are:

1. **Handling large amounts of text data**: LLMs need to handle vast amounts of text data, which can be difficult to process, store, and analyze.
2. **Understanding language complexity**: Human language is complex, with nuances, Context, and subtleties that need to be capture

2.3 self consistency

In [ ]:
from openai import OpenAI
import re, collections

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"

def cot_answer(question, temperature=0.5):
    prompt = f"""Answer the following question with step-by-step reasoning and only include the final number after
    **Therefore, (answer here)**.
    Question: {question}
    Let's think step by step:
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    content = response.choices[0].message.content
    match = re.search(r"[Tt]herefore,?\s*(.*)", content)
    return content, match.group(1).strip() if match else None


def self_consistent(question, n=10):
    answer = []
    for _ in range(n):
        _, ans = cot_answer(question)
        answer.append(ans)
    counter = collections.Counter(answer)
    winner, _ = counter.most_common(1)[0]
    return winner, counter


# Test cot_answer
question = "what is the square root of 169?"
full_answer, final_answer = cot_answer(question)
print("Full reasoning:\n", full_answer)
print("\nFinal answer:", final_answer)

# Test self_consistent
question = "what is the square root of 784?"
winner, counter = self_consistent(question)
print("Votes:", counter)
print("Chosen answer:", winner)

Full reasoning:
 To find the square root of 169, we can start by recalling that the square root of a number is a value that, when multiplied by itself, gives the original number.

Step 1: Look for perfect squares that are close to 169.
We know that 13^2 = 169, since 13 multiplied by itself equals 169.

Step 2: Check if there are any other perfect squares that are close to 169.
Since 13 is the only number whose square equals 169, we can conclude that the square root of 169 is 13.

Step 3: Write the final answer.
Therefore, (13).

Final answer: (13).
Votes: Counter({'(28).': 6, '(answer here)': 1, '(14).': 1, 'the square root of 784 is 4 * 7 = 28.': 1, 'the square root of 784 is 2^2 * 7, which is equal to 4 * 7 = 28.': 1})
Chosen answer: (28).


2.4 sequential revision

In [ ]:

from openai import OpenAI
client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"

def sequential_revision(question: str, max_steps: int = 5) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Keep your answers clean and correct"},
        {"role": "user", "content": question}
    ]

    draft = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.6
    ).choices[0].message.content.strip()

    for idx in range(max_steps):
        messages = [
            {"role": "system", "content": "You are a helpful assistant. Improve answers by making them cleaner and more accurate"},
            {"role": "user", "content": question},
            {"role": "assistant", "content": draft},
            {"role": "user", "content": "Please revise the answer. Make it cleaner, more accurate, and better written. Only include the new answer."}
        ]

        draft = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.6
        ).choices[0].message.content.strip()

        print(f"Draft {idx+1}:\n{draft}\n")

    return draft


# Test
question = "what is photosynthesis?"
final = sequential_revision(question, max_steps=3)
print("Final answer:\n", final)


Draft 1:
Photosynthesis is the process by which plants, algae, and some bacteria convert light energy from the sun into chemical energy in the form of organic compounds, such as glucose. This vital process is essential for life on Earth, as it produces oxygen and supports the food chain.

Photosynthesis requires three primary ingredients: water, carbon dioxide, and light energy. The process involves the absorption of water and carbon dioxide, followed by the conversion of light energy into chemical bonds, resulting in the production of glucose and oxygen.

The overall equation for photosynthesis is:

6 CO2 + 6 H2O + light energy → C6H12O6 (glucose) + 6 O2

This process is crucial for life, as it:

* Produces oxygen, which is necessary for the survival of most living organisms
* Provides energy for plants to grow and thrive
* Supports the food chain by serving as a source of energy and organic compounds for herbivores and decomposers

Draft 2:
Photosynthesis is the process by which plan

2.5 tree of thoughts

In [ ]:
# Word Ladder Puzzle

def neighbours(word, vocabulary):
  for i, c1 in enumerate(word):
    for c2 in 'abcdefghijklmnopqrstuvwxyz':
      if c1 != c2:
        candidate = word[:i] + c2 + word[i+1:]
        if candidate in vocabulary:
          yield candidate


def tree_of_thought(start, goal, vocab, max_depth=5, bean_width =4):
    frontier = [(start)]
    for depth in range(max_depth):
        candidate = []
        for path in frontier:
          for nxt in neighbours(path[-1], vocab):
            if nxt in path:
              continue
            candidate.append(path=[nxt])

    scored = sorted(candidate, key=lambda p: sum(a!=b for a,b in zip(p[-1], goal)))
    frontier = scored[:bean_width]
    if any(p[-1] == goal for p in frontier):
        return [p for p in frontier if p[-1] == goal][0]
    return None

vocab = {"hit","dot","cog","log","dog","lit","hot"}
print(tree_of_thought("hit", "cog", vocab))


None


In [ ]:
# Generic ToT Search

import re

model = "llama3.2:3b"

def propose_thoughts(question, state, k=2):

  prompt = f""" you are exploring solutions.
          problem: {question}
          current partial solutions: {state}

          propose at most {k} different next thoughts."""

  r = client.chat.completions.create(
      model=model,
      messages=[{"role": "user", "content": prompt}],
      temperature=0.5,
      n=k
  )
  return [c.message.content.strip() for choice in r.choices]

def score_state(question, state):
  prompt = f"""problem: {question}
      Rate from 1-10 how how promosing this partial solutions is : {state}
      """
  r = client.chat.completions.create(
      model=model,
      messages=[{"role": "user", "content": prompt}],
      temperature=0,
  )

4. deep research agent

In [ ]:
from ddgs import DDGS
from langchain.tools import Tool

def ddg_search(query: str, k: int = 5) -> str:
    # Use DDGS to run a simple web search and return joined snippets.
    with DDGS() as ddgs:
      results = [hit["body"] for hit in ddgs.text(query, max_results=k)]
      return "\n".join(results)

search_tool = Tool(
    name="DuckDuckGo Search",
    func=ddg_search,
    description="Search the public web. Input: a plain English query. Returns: concatenated snippets."
)

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain_community.chat_models import ChatOllama

MODEL = "deepseek-r1:8b"
question = "What are the best resources to learn machine learning in 2025?"

#step 1
llm = ChatOllama(model=MODEL, temperature = 0.2)

#step 2
agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

#step 3
result = agent.invoke({ "input": question })
print(result["output"])

/tmp/ipykernel_2256/2823146214.py:8: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=MODEL, temperature = 0.2)
/tmp/ipykernel_2256/2823146214.py:11: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_



> Entering new AgentExecutor chain...

Okay, let's plan ahead to 2025 for learning Machine Learning (ML). While specific courses and platforms evolve, the core concepts and foundational resources will remain relevant. Here's a breakdown of the best resources, considering the trajectory towards 2025:

## I. Foundational Knowledge (Essential for 2025)

1.  **Mathematics:**
    *   **Linear Algebra:** *Deep Learning with Python* (Chollet), *Essential Linear Algebra for Data Analysis and Machine Learning* (Kindle book). Khan Academy (Linear Algebra section).
    *   **Calculus (Differential + Integral):** *Calculus for Machine Learning* (free course by 3Blue1Brown on YouTube), *Mathematics for Machine Learning* (Coursera specialization by Google).
    *   **Probability & Statistics:** *StatQuest with Josh Starne* (YouTube channel), *Probability & Statistics* (Harvard Stat Online), *Statistics for Machine Learning* (free course by Stanford).
    *   **Optimization:** Concepts are covered 